# MuZero A/B Comparison: Old vs New Components

This notebook incrementally swaps old components with new ones to isolate which change breaks MuZero performance.

**How to use:**
1. Run Cell 0 (shared setup) first
2. Run Cell 1 (full OLD baseline) to establish working performance
3. Run each subsequent cell to test swapping in ONE new component
4. Compare metrics (loss curves, test scores) to identify the broken component

**Component groups being tested:**
- Cell 1: Full OLD system (baseline)
- Cell 2: Old + NEW WorldModel (DeterministicDynamics + ActionFusion)
- Cell 3: Old + NEW AgentNetwork (functional builder pattern, new head construction)
- Cell 4: Old + NEW Actors/Executors (RolloutActor + PettingZooAdapter)
- Cell 5: Old + NEW Learner/Registry (new pred_keys, new target builders)

In [1]:
# ============================================================
# CELL 0: SHARED SETUP (Run this first, always)
# ============================================================
import sys, os, time, importlib
import torch
import torch.nn.functional as F
import numpy as np

# Ensure project root is in path
# Find project root (works whether cwd is project root or old_muzero/)
cwd = os.getcwd()
if os.path.exists(os.path.join(cwd, "pyproject.toml")):
    PROJECT_ROOT = cwd
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DEVICE = torch.device("cpu")
TRAINING_STEPS = 10000  # Increase for more confident results
TEST_INTERVAL = 1000
TEST_TRIALS = 50

PARAMS = {
    "num_simulations": 25,
    "training_steps": TRAINING_STEPS,
    "transfer_interval": 1,
    "minibatch_size": 8,
    "replay_buffer_size": 100000,
    "num_workers": 4,  # Use 1 worker for reproducibility in local executor
    "num_envs_per_worker": 1,
    "num_puffer_threads": 1,
    "executor_type": "torch_mp",  # CRITICAL: Use local executor for notebook
    "search_backend": "python",
    "per_alpha": 0.0,
    "per_beta_schedule": {"type": "constant", "initial": 0.0},
    "temperature_schedule": {"type": "stepwise", "steps": [5], "values": [1.0, 0.0]},
    "n_step": 10,
    "dirichlet_alpha": 0.25,
    "known_bounds": [-1, 1],
    "action_selector": {
        "base": {"type": "categorical", "kwargs": {"exploration": True}}
    },
    "backbone": {
        "type": "resnet",
        "norm_type": "batch",
        "activation": "relu",
        "filters": [24] * 3,
        "kernel_sizes": [3] * 3,
        "strides": [1] * 3,
        "residual_layers": [(24, 3, 1)] * 3,
    },
    "reward_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "value_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
        "output_strategy": {"type": "scalar"},
    },
    "policy_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    "to_play_head": {
        "neck": {
            "type": "resnet",
            "filters": [16],
            "kernel_sizes": [1],
            "strides": [1],
            "norm_type": "batch",
        },
    },
    "stochastic": False,
    "gumbel": False,
    "search_batch_size": 5,
    "use_virtual_mean": True,
    "virtual_loss": 3.0,
    "atom_size": 1,
    "policy_loss_function": F.cross_entropy,
}

print(
    f"Setup complete. Training for {TRAINING_STEPS} steps, testing every {TEST_INTERVAL} steps."
)

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Setup complete. Training for 10000 steps, testing every 1000 steps.


In [ ]:
# ============================================================
# CELL 1: FULL OLD SYSTEM (Baseline) from cfd44e3
# ============================================================
# Uses: Old ModularAgentNetwork, Old MuzeroWorldModel, Old Actors,
#       Old Executor, Old Learner, Old Registry, Old Loss pred_keys
# ============================================================
from old_muzero.configs.games.tictactoe import TicTacToeConfig as OldTicTacToeConfig
from old_muzero.configs.agents.muzero import MuZeroConfig as OldMuZeroConfig
from old_muzero.agents.trainers.muzero_trainer import MuZeroTrainer as OldMuZeroTrainer
from old_muzero.stats.stats import StatTracker as OldStatTracker
from old_muzero.agents.random_agent import RandomAgent as OldRandomAgent
from old_muzero.agents.tictactoe_expert import (
    TicTacToeBestAgent as OldTicTacToeBestAgent,
)

torch.manual_seed(42)
np.random.seed(42)

old_game_config = OldTicTacToeConfig()
old_env = old_game_config.env_factory()
old_config = OldMuZeroConfig(PARAMS.copy(), old_game_config)

old_stats = OldStatTracker("old_baseline")
old_trainer = OldMuZeroTrainer(
    old_config,
    old_env,
    DEVICE,
    name="old_baseline",
    stats=old_stats,
    test_agents=[OldRandomAgent(), OldTicTacToeBestAgent()],
)
old_trainer.checkpoint_interval = 100
old_trainer.test_interval = TEST_INTERVAL
old_trainer.test_trials = TEST_TRIALS

start = time.time()
old_trainer.train()
print(f"\n=== OLD BASELINE completed in {time.time()-start:.1f}s ===")

In [ ]:
# ============================================================
# CELL 2: Old + NEW WorldModel (via WorldModelBridge)
# ============================================================
# WHAT CHANGED: Replaces old MuzeroWorldModel with WorldModelBridge.
# WorldModelBridge tests the NEW ActionFusion/SpatialEmbedding wired into
# the OLD ModularAgentNetwork + search + learner.
#
# What is NEW (under test): ActionFusion + SpatialActionEmbedding
# What is OLD (unchanged): Representation, Dynamics backbone, Reward/ToPlay heads
#
# WorldModelBridge lives in old_muzero/modules/world_models/bridge.py so that
# multiprocessing workers can import it (classes defined in notebooks can't be pickled).
# ============================================================
import numpy as np
import time
import torch

from old_muzero.configs.games.tictactoe import TicTacToeConfig as OldTicTacToeConfig
from old_muzero.configs.agents.muzero import MuZeroConfig as OldMuZeroConfig
from old_muzero.stats.stats import StatTracker as OldStatTracker
from old_muzero.agents.random_agent import RandomAgent as OldRandomAgent
from old_muzero.agents.tictactoe_expert import (
    TicTacToeBestAgent as OldTicTacToeBestAgent,
)
from old_muzero.agents.trainers.muzero_trainer import MuZeroTrainer as OldMuZeroTrainer

# Import from module (not defined inline) so multiprocessing workers can pickle it
from old_muzero.modules.world_models.bridge import WorldModelBridge

torch.manual_seed(42)
np.random.seed(42)

# --- Training Setup ---
# Pass WorldModelBridge via config_dict; trainer reads it and injects into ModularAgentNetwork
swap2_params = PARAMS.copy()
swap2_params["world_model_cls"] = WorldModelBridge

old_game_config = OldTicTacToeConfig()
old_env = old_game_config.env_factory()
old_config = OldMuZeroConfig(swap2_params, old_game_config)

swap2_stats = OldStatTracker("swap2_new_worldmodel")
swap2_trainer = OldMuZeroTrainer(
    old_config,
    old_env,
    DEVICE,
    name="swap2_new_worldmodel",
    stats=swap2_stats,
    test_agents=[OldRandomAgent(), OldTicTacToeBestAgent()],
)
swap2_trainer.checkpoint_interval = 100
swap2_trainer.test_interval = TEST_INTERVAL
swap2_trainer.test_trials = TEST_TRIALS

# === VERIFICATION: Confirm which WorldModel is actually in use ===
wm = swap2_trainer.agent_network.components["world_model"]
print(f"\n{'='*60}")
print("VERIFICATION")
print(f"  WorldModel class : {type(wm).__name__}")
print(f"  Is WorldModelBridge? {isinstance(wm, WorldModelBridge)}")
print(f'  Has new ActionFusion? {hasattr(wm, "action_fusion")}')
if hasattr(wm, "action_fusion"):
    print(f"  ActionFusion type   : {type(wm.action_fusion).__name__}")
    print(
        f"  Embedding module    : {type(wm.action_fusion.encoder.embedding_module).__name__}"
    )
print(f"  Representation shape: {wm.representation.output_shape}")
bridge_params = sum(p.numel() for p in wm.parameters())
print(f"  Bridge parameters   : {bridge_params:,}")
print(f"{'='*60}\n")

assert isinstance(wm, WorldModelBridge), (
    f"INJECTION FAILED: expected WorldModelBridge, got {type(wm).__name__}. "
    "Check that muzero_trainer.py reads world_model_cls from config_dict."
)

# --- Train ---
start = time.time()
swap2_trainer.train()
print(f"\n=== SWAP 2 (New ActionFusion) completed in {time.time()-start:.1f}s ===")

In [ ]:
# ============================================================
# CELL 3: Old + NEW AgentNetwork (includes new WorldModel + Search)
# ============================================================
# WHAT CHANGED:  Replaces OLD ModularAgentNetwork with NEW AgentNetwork
#               (built via build_agent_network).  Uses NEW search engine
#               (via NewSearchPettingZooActor) because the new network's
#               InferenceOutput.recurrent_state is incompatible with old search.
#
# What is NEW:  AgentNetwork, WorldModel, ActionFusion, heads (built by factory)
#               Search engine (new Python MCTS) wrapped in old SearchPolicySource
# What is OLD:  PettingZoo actor loop (NewSearchPettingZooActor subclass),
#               TorchMP executor, replay buffer, buffer format, base learner infra
# ============================================================
import time
import numpy as np
import torch

from configs.games.tictactoe import TicTacToeConfig as NewTicTacToeConfig
from configs.agents.muzero import MuZeroConfig as NewMuZeroConfig
from agents.factories.model import build_agent_network
from agents.factories.learner import build_universal_learner as new_build_learner

from old_muzero.agents.trainers.base_trainer import BaseTrainer as OldBaseTrainer
from old_muzero.agents.learner.batch_iterators import (
    SingleBatchIterator as OldSingleBatchIterator,
)
from old_muzero.replay_buffers.buffer_factories import (
    create_muzero_buffer as old_create_muzero_buffer,
)
from old_muzero.agents.action_selectors.selectors import CategoricalSelector
from old_muzero.agents.action_selectors.decorators import TemperatureSelector
from old_muzero.agents.executors.factory import create_executor as old_create_executor

# Must be imported from a module (not defined here) for multiprocessing pickling
from old_muzero.agents.workers.new_search_actor import (
    NewSearchPettingZooActor,
    NewSearchTester,
)

from old_muzero.stats.stats import StatTracker as OldStatTracker
from old_muzero.agents.random_agent import RandomAgent as OldRandomAgent
from old_muzero.agents.tictactoe_expert import (
    TicTacToeBestAgent as OldTicTacToeBestAgent,
)


class FixedToPlayIterator:
    """
    Wraps the old buffer's SingleBatchIterator and patches post-terminal
    to_play rows from [0,...,0] (old fill value) to uniform [1/N,...,1/N]
    so the new ToPlayLoss categorical-sum check doesn't fire.
    This is purely a Cell-3-local shim — no new code is modified.
    """

    def __init__(self, buffer, device, num_players):
        self._buffer = buffer
        self._device = device
        self._num_players = num_players

    def __iter__(self):
        batch = self._buffer.sample()
        batch = {
            k: (
                v.to(self._device, non_blocking=True).contiguous()
                if torch.is_tensor(v)
                else v
            )
            for k, v in batch.items()
        }
        if "to_plays" in batch:
            tp = batch["to_plays"]
            if tp.dim() >= 2:
                zero_mask = (tp.sum(dim=-1, keepdim=True) == 0).expand_as(tp)
                uniform = torch.full_like(tp, 1.0 / self._num_players)
                batch["to_plays"] = torch.where(zero_mask, uniform, tp)
        yield batch


class Swap3Trainer(OldBaseTrainer):
    """
    Old BaseTrainer skeleton with NEW AgentNetwork and NEW search.
    Actors use NewSearchPettingZooActor (old PettingZoo loop + new MCTS).
    Executor and buffer remain old.  Learner is new (matches new output keys).
    """

    def __init__(self, config, env, device, name="agent", stats=None, test_agents=None):
        super().__init__(config, env, device, name, stats, test_agents)

        if hasattr(env, "possible_agents"):
            self.player_id_mapping = {a: i for i, a in enumerate(env.possible_agents)}
        else:
            self.player_id_mapping = {"player_0": 0}

        # === NEW: AgentNetwork (auto-selects SpatialActionEmbedding for TicTacToe) ===
        self.agent_network = build_agent_network(
            config=config,
            obs_dim=self.obs_dim,
            num_actions=self.num_actions,
            device=device,
        )
        if config.kernel_initializer is not None:
            self.agent_network.initialize(config.kernel_initializer)
        if config.multi_process:
            self.agent_network.share_memory()

        # === OLD: Action selector ===
        inner = CategoricalSelector()
        self.action_selector = TemperatureSelector(inner_selector=inner, config=config)

        # === OLD: Replay buffer ===
        self.buffer = old_create_muzero_buffer(
            observation_dimensions=self.obs_dim,
            max_size=config.replay_buffer_size,
            num_actions=self.num_actions,
            num_players=config.game.num_players,
            player_id_mapping=self.player_id_mapping,
            unroll_steps=config.unroll_steps,
            n_step=config.n_step,
            gamma=config.discount_factor,
            batch_size=config.minibatch_size,
            observation_dtype=self.obs_dtype,
            alpha=config.per_alpha,
            beta=config.per_beta_schedule.initial,
            epsilon=config.per_epsilon,
            use_batch_weights=config.per_use_batch_weights,
            use_initial_max_priority=config.per_use_initial_max_priority,
            lstm_horizon_len=config.lstm_horizon_len,
            value_prefix=config.use_value_prefix,
            tau=config.reanalyze_tau,
            multi_process=config.multi_process,
            observation_quantization=config.observation_quantization,
            observation_compression=config.observation_compression,
        )

        # === OLD: TorchMP executor ===
        self.executor = old_create_executor(config)

        # === NEW: Learner (matches new AgentNetwork output keys) ===
        self.learner = new_build_learner(
            config=config,
            agent_network=self.agent_network,
            device=device,
            priority_update_fn=self.buffer.update_priorities,
            weight_broadcast_fn=self.executor.update_weights,
            validator_params={
                "minibatch_size": config.minibatch_size,
                "unroll_steps": config.unroll_steps,
                "num_actions": self.num_actions,
                "num_players": config.game.num_players,
                "atom_size": 1,
            },
        )

        if config.multi_process:
            self.buffer.share_memory()

        # === OLD executor with NEW actor (NewSearchPettingZooActor) ===
        # New config has env_factory (not env_factory), so pass it explicitly.
        worker_args = (
            config.game.env_factory,
            self.agent_network,
            self.action_selector,
            self.buffer,
            config.game.num_players,
            config,
            device,
            self.name,
        )
        self.actor_cls = NewSearchPettingZooActor
        self.executor.launch(self.actor_cls, worker_args, config.num_workers)

    def setup_tester(self):
        """Override: launch NewSearchTester so evaluation uses new search engine."""
        from old_muzero.agents.workers.tester import TestFactory
        from old_muzero.modules.utils import get_uncompiled_model

        test_types = []
        for agent in self.test_agents:
            for player_idx in range(self.config.game.num_players):
                from old_muzero.agents.workers.tester import VsAgentTest

                test_types.append(
                    VsAgentTest(
                        name=f"vs_{agent.name}_p{player_idx}",
                        num_trials=self.test_trials,
                        opponent=agent,
                        player_idx=player_idx,
                    )
                )
        from old_muzero.agents.workers.tester import SelfPlayTest

        test_types.append(SelfPlayTest(name="self_play", num_trials=self.test_trials))

        uncompiled = get_uncompiled_model(self.agent_network)
        launch_args = (
            self.config.game.env_factory,
            uncompiled,
            self.action_selector,
            None,
            self.config.game.num_players,
            self.config,
            torch.device("cpu"),
            f"{self.name}_tester",
            test_types,
        )
        self.executor.launch(NewSearchTester, launch_args, num_workers=1)

    def train(self):
        self._setup_stats()
        while self.training_step < self.config.training_steps:
            self.train_step()
            self.poll_test()
            if self.training_step % 100 == 0 and self.training_step > 0:
                print(f"Step {self.training_step}")
            if hasattr(self.stats, "drain_queue"):
                self.stats.drain_queue()
        self.stop_test()
        print("Training finished.")

    def train_step(self):
        _, collect_stats = self.executor.collect_data(
            min_samples=None, worker_type=self.actor_cls
        )
        for key, val in collect_stats.items():
            self.stats.append(key, val)

        if self.buffer.size >= self.config.min_replay_buffer_size:
            for _ in range(self.config.num_minibatches):
                iterator = FixedToPlayIterator(
                    self.buffer, self.device, self.config.game.num_players
                )
                for step_metrics in self.learner.step(iterator):
                    self._record_learner_metrics(step_metrics)
            self.training_step += 1
            if self.training_step % self.config.transfer_interval == 0:
                self.executor.update_weights(self.agent_network.state_dict())
            if self.training_step % self.checkpoint_interval == 0:
                self._save_checkpoint({})
            if self.training_step % self.test_interval == 0:
                self.trigger_test(self.agent_network.state_dict(), self.training_step)
                # Save plots so test scores are visible even if checkpoint_interval=inf
                import os
                from pathlib import Path

                graph_dir = Path("checkpoints", self.name, "graphs")
                os.makedirs(graph_dir, exist_ok=True)
                self.stats.plot_graphs(dir=graph_dir)
        return {}

    def _setup_stats(self):
        from old_muzero.stats.stats import PlotType

        test_score_keys = [f"vs_{a.name}_score" for a in self.test_agents]
        for key in [
            "score",
            "policy_loss",
            "value_loss",
            "reward_loss",
            "to_play_loss",
            "cons_loss",
            "loss",
            "test_score",
            "episode_length",
            "learner_fps",
            "actor_fps",
        ] + test_score_keys:
            if key not in self.stats.stats:
                if "_score" in key:
                    self.stats._init_key(
                        key,
                        subkeys=[f"p{i}" for i in range(self.config.game.num_players)]
                        + ["avg"],
                    )
                else:
                    self.stats._init_key(key)
        self.stats.add_plot_types(
            "score",
            PlotType.ROLLING_AVG,
            PlotType.BEST_FIT_LINE,
            rolling_window=100,
            ema_beta=0.6,
        )
        self.stats.add_plot_types(
            "test_score", PlotType.BEST_FIT_LINE, PlotType.VARIATION_FILL
        )
        for key in test_score_keys:
            self.stats.add_plot_types(
                key, PlotType.BEST_FIT_LINE, PlotType.VARIATION_FILL
            )


torch.manual_seed(42)
np.random.seed(42)

new_game_config = NewTicTacToeConfig()
new_env = new_game_config.env_factory()
new_config = NewMuZeroConfig(PARAMS.copy(), new_game_config)
# Old tester machinery calls config.game.env_factory — alias it to env_factory
if not hasattr(new_config.game, "env_factory"):
    new_config.game.env_factory = new_config.game.env_factory

try:
    swap3_stats = OldStatTracker("swap3_new_network")
    swap3_trainer = Swap3Trainer(
        new_config,
        new_env,
        DEVICE,
        name="swap3_new_network",
        stats=swap3_stats,
        test_agents=[OldRandomAgent(), OldTicTacToeBestAgent()],
    )
    swap3_trainer.checkpoint_interval = 100
    swap3_trainer.test_interval = TEST_INTERVAL
    swap3_trainer.test_trials = TEST_TRIALS

    # === VERIFICATION: Confirm new network is in use ===
    from modules.models.agent_network import AgentNetwork as NewAgentNetwork

    print(f"\n{'='*60}")
    print("VERIFICATION")
    print(f"  AgentNetwork class : {type(swap3_trainer.agent_network).__name__}")
    print(
        f"  Is new AgentNetwork? {isinstance(swap3_trainer.agent_network, NewAgentNetwork)}"
    )
    wm = (
        swap3_trainer.agent_network.components["world_model"]
        if "world_model" in swap3_trainer.agent_network.components
        else None
    )
    if wm is not None:
        print(f"  WorldModel class   : {type(wm).__name__}")
        dyn = getattr(wm, "dynamics_pipeline", None)
        if dyn:
            print(f"  Dynamics pipeline  : {type(dyn).__name__}")
            fusion = getattr(dyn, "dynamics_fusion", None)
            if fusion:
                print(
                    f"  ActionFusion emb   : {type(fusion.encoder.embedding_module).__name__}"
                )
    print(f"  Actor class        : {swap3_trainer.actor_cls.__name__}")
    print(f"{'='*60}\n")
    assert isinstance(swap3_trainer.agent_network, NewAgentNetwork), "INJECTION FAILED"

    start = time.time()
    swap3_trainer.train()
    print(
        f"\n=== SWAP 3 (New AgentNetwork + New Search) completed in {time.time()-start:.1f}s ==="
    )
except Exception as e:
    print(f"\n=== SWAP 3 FAILED: {e} ===")
    import traceback

    traceback.print_exc()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using         agent_type                    : muzero
Using         agent_type                    : muzero
Using default results_path                  : results
Using         training_steps                : 10000
Using default adam_epsilon                  : 1e-08
Using default momentum                      : 0.9
Using default learning_rate                 : 0.001
Using default clipnorm                      : 0
Using default optimizer                     : <class 'torch.optim.adam.Adam'>
Using default weight_decay                  : 0.0
Using default num_minibatches               : 1
Using default training_iterations           : 1
Using         lr_schedule                   : None
Using         lr_schedule                   : {'type': 'constant', 'initial': 0.001}
Using default test_interval                 : 1000
Using default checkpoint_interval           : 1000
Using         minibatch_size                : 8
Using         replay_buffer_size            : 100000
Using default min_repla

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report 

Initializing stat 'num_episodes' with subkeys None
Initializing stat 'mean_is_weight' with subkeys None
Initializing stat 'max_is_weight' with subkeys None
Initializing stat 'min_is_weight' with subkeys None
Initializing stat 'ValueLoss_unweighted' with subkeys None
Initializing stat 'ValueLoss' with subkeys None
Initializing stat 'approx_kl' with subkeys None
Initializing stat 'PolicyLoss_unweighted' with subkeys None
Initializing stat 'PolicyLoss' with subkeys None
Initializing stat 'RewardLoss_unweighted' with subkeys None
Initializing stat 'RewardLoss' with subkeys None
Initializing stat 'ToPlayLoss_unweighted' with subkeys None
Initializing stat 'ToPlayLoss' with subkeys None
Initializing stat 'state_value/mean' with subkeys None
Initializing stat 'policy_logits/entropy' with subkeys None
Initializing stat 'loss_pipeline_latency_ms' with subkeys None
Initializing stat 'learner_throughput' with subkeys None
plotting score
plotting loss
plotting episode_length
plotting actor_fps
plo

/Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Saved stats plot to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/old_muzero/checkpoints/swap3_new_network/graphs/swap3_new_network_stats.png
Step 1000


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[WARNING]: Illegal move made, game terminating with current player losing. 
obs['action_mask'] contains a mask of all legal moves that can be chosen.
plotting score
plotting loss
plotting episode_length
plotting actor_fps
plotting num_episodes
plotting mean_is_weight
plotting max_is_weight
plotting min_is_weight
plotting ValueLoss_unweighted
plotting ValueLoss
plotting approx_kl
plotting PolicyLoss_unweighted
plotting PolicyLoss
plotting RewardLoss_unweighted
plotting RewardLoss
plotting ToPlayLoss_unweighted
plotting ToPlayLoss
plotting state_value/mean
plotting policy_logits/entropy
plotting loss_pipeline_latency_ms
plotting learner_throughput
Saved stats plot to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/old_muzero/checkpoints/swap3_new_network/graphs/swap3_new_network_stats.png
Saved checkpoint at step 1100 to /Users/jonathanlamontange-kratz/Documents/GitHub/rl-stuff/old_muzero/checkpoints/swap3_new_network/step_1100
Step 1100
[vs_random_p0] score: 0.380 (step 1000)


In [ ]:
# ============================================================
# CELL 4: Old + NEW Actors / Executor / Adapters
# ============================================================
# WHAT CHANGED:  Replaces OLD TorchMPExecutor + PettingZooActor with NEW
#               TorchMPExecutor + RolloutActor + PettingZooAdapter.
#               Uses the NEW MuZeroTrainer which wires all of this.
#               Evaluators are intentionally DISABLED (test_interval=inf)
#               so that Cell 5 can test them in isolation.
#
# What is NEW:  RolloutActor, PettingZooAdapter, TorchMPExecutor (new),
#               AgentNetwork, WorldModel, search engine
# What is OLD:  Nothing — this is a full new-stack training run without evaluators
# ============================================================
import time
import numpy as np
import torch

from configs.games.tictactoe import TicTacToeConfig as NewTicTacToeConfig
from configs.agents.muzero import MuZeroConfig as NewMuZeroConfig
from agents.trainers.muzero_trainer import MuZeroTrainer as NewMuZeroTrainer
from stats.stats import StatTracker as NewStatTracker
from old_muzero.agents.random_agent import RandomAgent as OldRandomAgent
from old_muzero.agents.tictactoe_expert import (
    TicTacToeBestAgent as OldTicTacToeBestAgent,
)

torch.manual_seed(42)
np.random.seed(42)

new_game_config4 = NewTicTacToeConfig()
new_env4 = new_game_config4.env_factory()
new_config4 = NewMuZeroConfig(PARAMS.copy(), new_game_config4)

try:
    swap4_stats = NewStatTracker("swap4_new_actors")
    # test_agents=[] disables evaluator setup inside the trainer
    swap4_trainer = NewMuZeroTrainer(
        new_config4,
        new_env4,
        DEVICE,
        name="swap4_new_actors",
        stats=swap4_stats,
        test_agents=[],
    )
    swap4_trainer.checkpoint_interval = 999999
    swap4_trainer.test_interval = 999999  # Evaluators tested separately in Cell 5
    swap4_trainer.test_trials = TEST_TRIALS

    from agents.workers.actors import RolloutActor
    from agents.environments.adapters import PettingZooAdapter

    print(f"\n{'='*60}")
    print("VERIFICATION")
    print(f"  Executor type      : {type(swap4_trainer.executor).__name__}")
    print(f"  Actor class        : RolloutActor (new)")
    print(f"  Adapter class      : PettingZooAdapter (new)")
    print(f"  AgentNetwork class : {type(swap4_trainer.agent_network).__name__}")
    print(f"{'='*60}\n")

    start = time.time()
    swap4_trainer.train()
    print(
        f"\n=== SWAP 4 (New Actors/Executor/Adapters) completed in {time.time()-start:.1f}s ==="
    )
except Exception as e:
    print(f"\n=== SWAP 4 FAILED: {e} ===")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================
# CELL 5: NEW Evaluators / Testers (standalone)
# ============================================================
# WHAT THIS TESTS:  The new EvaluatorActor + test infrastructure in isolation.
#                   Old system used a list of test_agents; new system uses
#                   BaseTestType subclasses (SelfPlayTest, VsAgentTest).
#
# Run AFTER Cell 4.  Uses swap4_trainer.agent_network and search.
# Evaluators behave very differently from old test_agents, so this cell
# isolates them so regressions here don't contaminate training results.
# ============================================================
import time
import numpy as np
import torch

from agents.workers.evaluator import EvaluatorActor, SelfPlayTest, VsAgentTest
from agents.environments.adapters import PettingZooAdapter
from agents.action_selectors.selectors import ArgmaxSelector, LegalMovesMaskDecorator
from agents.action_selectors.policy_sources import SearchPolicySource
from agents.factories.search import SearchBackendFactory

try:
    # Re-use swap4_trainer network + config (must run Cell 4 first)
    assert "swap4_trainer" in dir(), "Run Cell 4 first"

    eval_config = new_config4
    eval_network = swap4_trainer.agent_network
    eval_device = DEVICE
    eval_num_actions = eval_config.game.num_actions
    eval_num_players = eval_config.game.num_players

    # Build search engine for evaluation (same config as training)
    eval_search = SearchBackendFactory.create(
        eval_config, device=eval_device, num_actions=eval_num_actions
    )
    eval_policy_source = SearchPolicySource(
        search_engine=eval_search,
        agent_network=eval_network,
    )

    # Action selector for evaluation (argmax, legal-move aware)
    eval_selector = LegalMovesMaskDecorator(ArgmaxSelector())

    # Test suite: self-play only (VsAgentTest requires opponents with obs_inference API)
    test_types = [
        SelfPlayTest("self_play", num_trials=TEST_TRIALS),
    ]

    evaluator = EvaluatorActor(
        adapter_cls=PettingZooAdapter,
        adapter_args=(eval_config.game.env_factory,),
        network=eval_network,
        policy_source=eval_policy_source,
        buffer=None,
        action_selector=eval_selector,
        actor_device=str(eval_device),
        num_actions=eval_num_actions,
        num_players=eval_num_players,
        test_types=test_types,
    )

    print(f"Running {TEST_TRIALS} self-play episodes...")
    start = time.time()
    results = evaluator.run_tests(num_trials=TEST_TRIALS)
    elapsed = time.time() - start

    print(f"\n{'='*60}")
    print("EVALUATOR RESULTS")
    for test_name, metrics in results.items():
        if isinstance(metrics, dict):
            print(f"  [{test_name}]")
            for k, v in metrics.items():
                if isinstance(v, float):
                    print(f"    {k}: {v:.4f}")
    print(f"  Elapsed: {elapsed:.1f}s")
    print(f"{'='*60}\n")

except Exception as e:
    print(f"\n=== CELL 5 (Evaluators) FAILED: {e} ===")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================
# CELL 6: Learner Throughput Benchmark
# ============================================================
# WHAT THIS TESTS:  Old vs New learner steps/second on identical fake batches.
#                   The new learner reportedly has 2x throughput — this cell
#                   verifies whether that's a genuine speed-up or a sign that
#                   the new learner is doing less work (e.g. skipping loss terms).
#
# Runs in the MAIN PROCESS with no workers, so results are comparable.
# ============================================================
import time
import torch
import numpy as np

BENCH_STEPS = 200
WARMUP_STEPS = 20

torch.manual_seed(42)
np.random.seed(42)


def make_fake_batch(config, obs_dim, num_actions, device, num_players=2):
    """Creates a fake MuZero batch with the same keys as the real buffer."""
    B = config.minibatch_size
    T = config.unroll_steps + 1
    batch = {
        "observations": torch.zeros(B, T, *obs_dim, device=device),
        "actions": torch.zeros(B, T, device=device),
        "rewards": torch.zeros(B, T, device=device),
        "values": torch.zeros(B, T, device=device),
        "policies": torch.ones(B, T, num_actions, device=device) / num_actions,
        "to_plays": torch.zeros(B, T, device=device, dtype=torch.int16),
        "weights": torch.ones(B, device=device),
        "ids": torch.arange(B, device=device, dtype=torch.int64),
        "game_ids": torch.arange(B, device=device, dtype=torch.int64),
        "training_steps": torch.zeros(B, device=device, dtype=torch.int64),
    }
    return batch


def benchmark_learner(learner, batch, steps, warmup):
    """Returns avg steps/sec over `steps` iterations (after `warmup` warm-up steps)."""

    class FakeBatchIterator:
        def __iter__(self):
            yield {k: v.clone() for k, v in batch.items()}

    # Warm up
    for _ in range(warmup):
        for _ in learner.step(FakeBatchIterator()):
            pass

    start = time.perf_counter()
    for _ in range(steps):
        for _ in learner.step(FakeBatchIterator()):
            pass
    elapsed = time.perf_counter() - start
    return steps / elapsed


try:
    # --- OLD learner ---
    from old_muzero.configs.games.tictactoe import TicTacToeConfig as OldTicTacToeConfig
    from old_muzero.configs.agents.muzero import MuZeroConfig as OldMuZeroConfig
    from old_muzero.agents.learner.factory import (
        build_universal_learner as old_build_learner,
    )
    from old_muzero.modules.agent_nets.modular import (
        ModularAgentNetwork as OldModularAgentNetwork,
    )

    old_game_cfg = OldTicTacToeConfig()
    old_cfg_bench = OldMuZeroConfig(PARAMS.copy(), old_game_cfg)
    old_env_bench = old_game_cfg.env_factory()

    obs_dim_bench = swap3_trainer.obs_dim  # reuse from Cell 3
    num_actions_bench = old_game_cfg.num_actions

    old_net_bench = OldModularAgentNetwork(
        config=old_cfg_bench,
        input_shape=obs_dim_bench,
        num_actions=num_actions_bench,
    ).to(DEVICE)

    old_learner_bench = old_build_learner(
        config=old_cfg_bench,
        agent_network=old_net_bench,
        device=DEVICE,
        priority_update_fn=None,
        weight_broadcast_fn=None,
    )

    # --- NEW learner ---
    from configs.games.tictactoe import TicTacToeConfig as NewTicTacToeConfig
    from configs.agents.muzero import MuZeroConfig as NewMuZeroConfig
    from agents.factories.model import build_agent_network
    from agents.factories.learner import build_universal_learner as new_build_learner

    new_game_cfg_bench = NewTicTacToeConfig()
    new_cfg_bench = NewMuZeroConfig(PARAMS.copy(), new_game_cfg_bench)
    new_env_bench = new_game_cfg_bench.env_factory()

    new_net_bench = build_agent_network(
        new_cfg_bench, obs_dim_bench, num_actions_bench, DEVICE
    )
    new_learner_bench = new_build_learner(
        config=new_cfg_bench,
        agent_network=new_net_bench,
        device=DEVICE,
        priority_update_fn=None,
        weight_broadcast_fn=None,
        validator_params={
            "minibatch_size": new_cfg_bench.minibatch_size,
            "unroll_steps": new_cfg_bench.unroll_steps,
            "num_actions": num_actions_bench,
            "num_players": new_game_cfg_bench.num_players,
            "atom_size": 1,
        },
    )

    fake_batch = make_fake_batch(
        old_cfg_bench, obs_dim_bench, num_actions_bench, DEVICE
    )

    print(
        f"Benchmarking learners ({WARMUP_STEPS} warmup + {BENCH_STEPS} timed steps)..."
    )
    old_sps = benchmark_learner(
        old_learner_bench, fake_batch, BENCH_STEPS, WARMUP_STEPS
    )
    print(f"  OLD learner: {old_sps:.1f} steps/sec")

    new_sps = benchmark_learner(
        new_learner_bench, fake_batch, BENCH_STEPS, WARMUP_STEPS
    )
    print(f"  NEW learner: {new_sps:.1f} steps/sec")

    ratio = new_sps / old_sps
    print(f"\n  Ratio (new/old): {ratio:.2f}x")
    if ratio > 1.5:
        print(
            "  ⚠  New learner is >1.5x faster — check for skipped loss terms or missing grad steps."
        )
    elif ratio < 0.7:
        print("  ⚠  New learner is slower — possible regression in loss computation.")
    else:
        print("  ✓  Throughput is comparable.")

    # --- Loss term audit ---
    print("\nOld learner loss keys:")
    for _ in old_learner_bench.step([fake_batch]):
        pass  # primed; inspect modules
    print(
        f"  Pipeline modules: {[type(m).__name__ for m in getattr(old_learner_bench, 'loss_pipeline', {}).modules if hasattr(getattr(old_learner_bench, 'loss_pipeline', None), 'modules')]}"
    )

    print("\nNew learner loss keys:")
    print(
        f"  Pipeline modules: {[type(m).__name__ for m in getattr(new_learner_bench, 'loss_pipeline', {}).modules if hasattr(getattr(new_learner_bench, 'loss_pipeline', None), 'modules')]}"
    )

except Exception as e:
    print(f"\n=== CELL 6 (Throughput Benchmark) FAILED: {e} ===")
    import traceback

    traceback.print_exc()

In [ ]:
# ============================================================
# CELL 7: FULL NEW SYSTEM (for final comparison)
# ============================================================
# Uses: New AgentNetwork, New WorldModel, New RolloutActor,
#       New Executor, New Learner, New Evaluator
# This should match the current production behaviour.
# ============================================================
import time
import numpy as np
import torch

from configs.games.tictactoe import TicTacToeConfig as NewTicTacToeConfig
from configs.agents.muzero import MuZeroConfig as NewMuZeroConfig
from agents.trainers.muzero_trainer import MuZeroTrainer as NewMuZeroTrainer
from stats.stats import StatTracker as NewStatTracker
from old_muzero.agents.random_agent import RandomAgent as OldRandomAgent
from old_muzero.agents.tictactoe_expert import (
    TicTacToeBestAgent as OldTicTacToeBestAgent,
)

torch.manual_seed(42)
np.random.seed(42)

new_game_config7 = NewTicTacToeConfig()
new_env7 = new_game_config7.env_factory()
new_config7 = NewMuZeroConfig(PARAMS.copy(), new_game_config7)

try:
    new_stats7 = NewStatTracker("new_full")
    new_trainer7 = NewMuZeroTrainer(
        new_config7,
        new_env7,
        DEVICE,
        name="new_full",
        stats=new_stats7,
        test_agents=[],  # New evaluator runs via Cell 5 pattern; old agents incompatible with new VsAgentTest
    )
    new_trainer7.checkpoint_interval = 999999
    new_trainer7.test_interval = TEST_INTERVAL
    new_trainer7.test_trials = TEST_TRIALS

    start = time.time()
    new_trainer7.train()
    print(f"\n=== NEW FULL completed in {time.time()-start:.1f}s ===")
except Exception as e:
    print(f"\n=== NEW FULL FAILED: {e} ===")
    import traceback

    traceback.print_exc()

## How to interpret results

| Cell | What is NEW | Expected if component is OK | Expected if component is broken |
|------|------------|---------------------------|-------------------------------|
| 1 | Nothing (full OLD baseline) | Losses decrease, scores improve | N/A |
| 2 | WorldModel (Bridge: new ActionFusion + SpatialEmbedding) | Same as Cell 1 | Representation issues |
| 3 | AgentNetwork + WorldModel + new search (old actors/executor) | Same as Cell 2 | Network routing / head issues |
| 4 | Actors + Executor + Adapters (RolloutActor + PettingZooAdapter) | Same as Cell 3 | Collection or env loop issues |
| 5 | Evaluators (EvaluatorActor + SelfPlayTest) | Reasonable self-play scores | Eval bugs, wrong action selection |
| 6 | Learner throughput (benchmark) | ~1x ratio vs old | >1.5x → missing loss terms; <0.7x → regression |
| 7 | Full new system | Matches training curve of Cell 4 | Any remaining issues |

**Diagnostic strategy:** The first cell that diverges from the prior cell's behaviour contains the regression.
